# 05 · Results: tables, paired tests, leakage and figures

In this notebook I read the fold scores written by `04_modeling` and produce every table and figure in the README, together with `results/results.json`, which holds every number the README quotes. Nothing here refits a model, so it runs in seconds.

The commentary describes the full run (3 repeats × 5 folds, three fold schemes). With `CODON_SMOKE=1` the notebook reads the tiny design in `results/smoke/` instead, and writes its figures there; in that case the tables are shorter, the numbers differ a little, and the leakage comparison is skipped.

In [ ]:
import ast
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import codon_utils

plt.rcParams['figure.dpi'] = 72  # small inline images; the PNG files are saved at 150 dpi

SMOKE = os.environ.get('CODON_SMOKE') == '1'
RES = Path('../results/smoke' if SMOKE else '../results')
FIG = RES / 'figures' if SMOKE else Path('../figures')
FIG.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(RES / 'fold_scores.csv')
headline = {}  # every number the README quotes; written to results.json at the end
print(len(scores), 'fold scores read from', RES)

## Mean and spread over folds

First, the mean ± standard deviation of each metric over the 15 outer test folds. The table shows the main scheme, with species held out; `results/summary_by_scheme.csv` has all three.

In [ ]:
summary = codon_utils.summarize(scores)
summary.to_csv(RES / 'summary_by_scheme.csv', index=False, float_format='%.6f')
for r in summary.itertuples():
    for m in codon_utils.METRICS:
        key = f'summary.{r.scheme}.{r.task}.{r.features}.{r.model}.{m}'
        headline[f'{key}.mean'], headline[f'{key}.sd'] = getattr(r, f'{m}_mean'), getattr(r, f'{m}_sd')


def mean_sd_table(scheme):
    s = summary[summary['scheme'] == scheme]
    cells = {m: s[f'{m}_mean'].map('{:.3f}'.format) + ' ± ' + s[f'{m}_sd'].map('{:.3f}'.format)
             for m in codon_utils.METRICS}
    return pd.DataFrame(cells).set_index(pd.MultiIndex.from_frame(s[['task', 'features', 'model']]))


mean_sd_table('species')

A few things stand out. DNA type is predicted almost perfectly: four models reach a macro-F1 of 0.98–0.99 with codon features. Kingdom is harder but still well predicted: k-NN leads with about 0.88, the MLP and XGBoost are close behind (0.87 and 0.86), and the random forest is some way back (0.81). The majority-class baseline, which always predicts 'bacteria', scores 0.04 on kingdom and 0.28 on DNA type.

Naive Bayes trails everywhere, as one would expect: its assumption that the features are independent within each class is badly violated here, since a row's codon frequencies sum to one and are strongly correlated. Curiously, k-NN has the best macro-F1 for kingdom, but the *lowest* ROC-AUC of the strong models (0.94 against about 0.99). The inner search usually chooses k = 1 (see *Tuning choices* below), which gives sharp labels but probabilities that are nearly all 0 or 1, and AUC rewards a well-graded ranking. If calibrated probabilities mattered, I'd use XGBoost or the MLP.

## Is the best model better than the others?

For each scheme, task and feature set, I compare the model with the highest mean macro-F1 with every other model, on the same folds. The 15 fold scores, however, are not independent, because the training sets overlap heavily (any two of them share most of their rows), so the differences between two models are positively correlated from fold to fold, and a plain paired t-test underestimates their variance. The Nadeau–Bengio corrected test allows for this by replacing the variance of the mean difference, s²/J, with (1/J + n_test/n_train)·s². I show the plain paired test alongside, for contrast.

In [ ]:
def paired(sub, by, a, b):
    # scores of a and b on identical folds, in the same order
    x = sub[sub[by] == a].sort_values(['repeat', 'fold'])
    y = sub[sub[by] == b].sort_values(['repeat', 'fold'])
    assert (x[['repeat', 'fold', 'n_test']].to_numpy() == y[['repeat', 'fold', 'n_test']].to_numpy()).all()
    return x, y


def compare(x, y, metric):
    n_train, n_test = x['n_train'].mean(), x['n_test'].mean()
    diff, t, p = codon_utils.corrected_ttest(x[metric], y[metric], n_train, n_test)
    lo, hi = codon_utils.corrected_ci(x[metric], y[metric], n_train, n_test)
    return {'diff': diff, 'ci95_lo': lo, 'ci95_hi': hi, 't_corrected': t, 'p_corrected': p,
            'p_naive_paired': stats.ttest_rel(x[metric], y[metric]).pvalue, 'J': len(x)}


rows = []
models_only = scores[scores['model'] != 'majority']
for (scheme, task, features), sub in models_only.groupby(['scheme', 'task', 'features'], sort=False):
    ranking = sub.groupby('model')['f1_macro'].mean().sort_values(ascending=False)
    headline[f'best.{scheme}.{task}.{features}'] = ranking.index[0]
    for rank, other in enumerate(ranking.index[1:], start=2):
        x, y = paired(sub, 'model', ranking.index[0], other)
        for metric in ['f1_macro', 'balanced_accuracy', 'roc_auc_ovr_macro']:
            rows.append({'scheme': scheme, 'task': task, 'features': features, 'model_a': ranking.index[0],
                         'model_b': other, 'rank_b': rank, 'metric': metric, **compare(x, y, metric)})
tests = pd.DataFrame(rows)
tests.to_csv(RES / 'best_vs_others.csv', index=False, float_format='%.6f')
for r in tests.itertuples():
    key = f'tests.{r.scheme}.{r.task}.{r.features}.{r.model_a}_vs_{r.model_b}.{r.metric}'
    for k in ['diff', 'ci95_lo', 'ci95_hi', 'p_corrected', 'p_naive_paired']:
        headline[f'{key}.{k}'] = getattr(r, k)

top2 = tests[tests['rank_b'] == 2]
top2.to_csv(RES / 'top2_tests.csv', index=False, float_format='%.6f')
cols = ['scheme', 'task', 'features', 'model_a', 'model_b', 'diff', 'ci95_lo', 'ci95_hi', 'p_corrected', 'p_naive_paired']
top2.loc[top2['metric'] == 'f1_macro', cols].round(4)

With species held out, the leading model is not significantly ahead of the runner-up on macro-F1, for any task or feature set. For kingdom with codons, for example, k-NN leads the MLP by about 0.011: the plain paired test gives p = 0.013, which would usually be reported as a win, and the corrected test gives p = 0.21. The plain test would call two of these four differences significant at the 5% level.

## Codon or amino-acid features?

This was the original question of the project. The same models on the same folds, with 64 codon frequencies against 21 amino-acid groups; positive differences favour the codons, and the intervals are corrected 95% confidence intervals.

In [ ]:
rows = []
for (scheme, task, model), sub in models_only.groupby(['scheme', 'task', 'model'], sort=False):
    x, y = paired(sub, 'features', 'codon', 'amino')
    for metric in ['f1_macro', 'roc_auc_ovr_macro']:
        rows.append({'scheme': scheme, 'task': task, 'model': model, 'metric': metric,
                     'codon': x[metric].mean(), 'amino': y[metric].mean(), **compare(x, y, metric)})
cva = pd.DataFrame(rows)
cva.to_csv(RES / 'codon_vs_amino.csv', index=False, float_format='%.6f')
for r in cva.itertuples():
    for k in ['codon', 'amino', 'diff', 'ci95_lo', 'ci95_hi', 'p_corrected']:
        headline[f'cva.{r.scheme}.{r.task}.{r.model}.{r.metric}.{k}'] = getattr(r, k)

species_f1 = cva[(cva['scheme'] == 'species') & (cva['metric'] == 'f1_macro')]
species_f1[['task', 'model', 'codon', 'amino', 'diff', 'ci95_lo', 'ci95_hi', 'p_corrected']].round(4)

In [ ]:
# Figure: codon - amino macro-F1 per task and model, with corrected 95% CIs
COLORS = ['#2a78d6', '#eb6834', '#1baf7a']  # fixed categorical order, checked for colour-blind separation
LABEL = {'gnb': 'Gaussian NB', 'knn': 'k-NN', 'rf': 'Random forest', 'xgb': 'XGBoost', 'mlp': 'MLP'}
TASK = {'kingdom': 'Kingdom', 'dna_type': 'DNA type'}

fig, ax = plt.subplots(figsize=(6.4, 0.34 * len(species_f1) + 1.4))
for i, r in enumerate(species_f1.itertuples()):
    color = COLORS[list(TASK).index(r.task)]
    ax.plot([r.ci95_lo, r.ci95_hi], [i, i], color=color, linewidth=2, solid_capstyle='round')
    ax.plot(r.diff, i, 'o', ms=6, color=color, mec='white', mew=1.5)
ax.axvline(0, color='#52514e', linewidth=1)
ax.set_yticks(range(len(species_f1)), [f'{TASK[r.task]}: {LABEL[r.model]}' for r in species_f1.itertuples()])
ax.invert_yaxis()
ax.grid(axis='x', color='#e4e3df')
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlabel('codon − amino-acid macro-F1 (paired, corrected 95% CI)')
ax.set_title('64 codon vs 21 amino-acid frequencies, species held out', loc='left', fontsize=10)
fig.tight_layout()
fig.savefig(FIG / 'codon_vs_amino.png', dpi=150)
plt.show()

For kingdom, the codons win clearly: by 0.04 (random forest) to 0.12 (MLP) in macro-F1, and every corrected interval for the four strong models lies above zero. Summing synonymous codons throws away codon-usage bias (the preference, within a group of synonyms, for some codons over others), and that preference evidently carries a good deal of lineage information. For DNA type the advantage is small, 0.005 to 0.011, because the compartments differ in which amino acids (or rather, as `02_eda` showed, which codon groups) they use, and the groups keep that.

Naive Bayes is the exception: on DNA type it does markedly better with the 21 groups than with the 64 codons (0.92 against 0.83), presumably because fewer, less redundant features violate its independence assumption less badly.

So the 2022 conclusion, that amino-acid features give "nearly comparable results with 67% fewer features", holds for DNA type, but not for kingdom.

## Leakage: what the test folds hold out

How much do these scores depend on how the test folds are built? The same models and tuning, under three schemes: random rows (*ungrouped*), whole species held out, and whole genus groups held out. The differences are between each scheme's mean macro-F1 over its 15 folds.

In [ ]:
schemes = [s for s in ['ungrouped', 'species', 'genus'] if s in set(scores['scheme'])]
f1 = summary[summary['model'] != 'majority'].pivot_table(
    index=['task', 'features', 'model'], columns='scheme', values='f1_macro_mean', sort=False)
diffs = {'ungrouped_minus_species': ('ungrouped', 'species'), 'species_minus_genus': ('species', 'genus')}
STRONG = ['knn', 'rf', 'xgb', 'mlp']
if len(schemes) < 3:
    print('leakage comparison needs all three schemes; skipped in a smoke run')
else:
    leak = f1[schemes].copy()
    for name, (a, b) in diffs.items():
        leak[name] = f1[a] - f1[b]
        for (task, features, model), v in leak[name].items():
            headline[f'leak.{task}.{features}.{model}.f1_macro_{name}'] = v
        for task, d in leak[name].groupby(level='task'):
            strong = d[d.index.get_level_values('model').isin(STRONG)]
            headline[f'leak_range.{task}.f1_macro_{name}.min'] = d.min()
            headline[f'leak_range.{task}.f1_macro_{name}.max'] = d.max()
            headline[f'leak_range_strong.{task}.f1_macro_{name}.min'] = strong.min()
            headline[f'leak_range_strong.{task}.f1_macro_{name}.max'] = strong.max()
    leak.to_csv(RES / 'leakage_comparison.csv', float_format='%.6f')
    display(leak.round(3))

In [ ]:
# Figure: macro-F1 (mean ± SD over folds) by model and fold scheme, codon features
SCHEME = {'ungrouped': 'random folds (ungrouped)', 'species': 'species held out', 'genus': 'genus held out'}
MARKER = {'ungrouped': 'o', 'species': 's', 'genus': 'D'}
codon = summary[(summary['features'] == 'codon') & summary['model'].isin(LABEL)]
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.9))
for ax, task in zip(axes, TASK):
    models = [m for m in LABEL if m in set(codon['model'])]
    x = np.arange(len(models))
    for scheme in schemes:
        i = list(SCHEME).index(scheme)  # colour and position follow the scheme, not its rank
        s = codon[(codon['task'] == task) & (codon['scheme'] == scheme)].set_index('model').reindex(models)
        ax.errorbar(x - 0.27 + 0.27 * i, s['f1_macro_mean'], yerr=s['f1_macro_sd'], fmt=MARKER[scheme],
                    ms=5, color=COLORS[i], elinewidth=1.5, label=SCHEME[scheme])
    ax.set_xticks(x, [LABEL[m].replace(' ', '\n') for m in models], fontsize=8.5)
    ax.set_title(TASK[task], loc='left', fontsize=10)
    ax.set_ylabel('macro-F1, mean ± SD over folds')
    ax.grid(axis='y', color='#e4e3df')
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(FIG / 'leakage_by_scheme.png', dpi=150)
plt.show()

Holding out species instead of random rows changes almost nothing: every difference is within ±0.005. As `03_preprocessing` showed, rows that share a `SpeciesID` are different compartments of one organism, no more alike than random pairs, so this apparent source of leakage is, in practice, negligible.

Holding out whole genus groups matters a great deal. For kingdom, the four strong models lose between 0.065 and 0.128 in macro-F1, k-NN and the random forest most of all; for DNA type, the loss is at most 0.011. That asymmetry makes sense. A compartment leaves a signature that is shared across lineages (mitochondria everywhere have their distinctive codon usage, and animal mitochondria their distinctive genetic code), so a model can recognise it in a genus it has never seen. Kingdom is a property of the lineage itself, and when the closest relatives are removed from training, the nearest-neighbour style of reasoning (find the most similar organism and copy its label) loses most of what it was relying on.

The ranking changes too: with genus held out, k-NN no longer has the highest mean kingdom score (the MLP does, with codon features, and XGBoost with amino-acid groups). The next table tests whether that change of leader is real.

In [ ]:
genus = tests[(tests['scheme'] == 'genus') & (tests['task'] == 'kingdom') & (tests['metric'] == 'f1_macro')]
genus[['features', 'model_a', 'model_b', 'diff', 'ci95_lo', 'ci95_hi', 'p_corrected']].round(4)

With genus held out, the MLP leads k-NN on kingdom (codon features) by about 0.02, but the corrected 95% interval includes zero (p = 0.09). The ordering, then, is no longer clear; the data do not show that the MLP is better; what *is* clear is that every model's kingdom score falls substantially once close relatives are out of reach.

## Where kingdom predictions go wrong

The row-normalised confusion matrix of the best kingdom model (codon features, species held out), pooling the out-of-fold predictions of all three repeats: each row shows where the members of one true class ended up.

In [ ]:
KINGDOMS = ['archaea', 'bacteria', 'bacteriophage', 'virus', 'plant',
            'invertebrate', 'vertebrate', 'mammal', 'rodent', 'primate']
NESTED = {'vertebrate', 'mammal', 'rodent', 'primate'}
confusion = pd.read_csv(RES / 'confusion_counts.csv')
best = headline['best.species.kingdom.codon']
cc = confusion[(confusion['task'] == 'kingdom') & (confusion['features'] == 'codon') & (confusion['model'] == best)]
counts = cc.pivot_table(index='y_true', columns='y_pred', values='count', aggfunc='sum', fill_value=0)
counts = counts.reindex(index=KINGDOMS, columns=KINGDOMS, fill_value=0)
share = counts.div(counts.sum(axis=1), axis=0)

for k, v in pd.Series(np.diag(share), index=KINGDOMS).items():
    headline[f'recall.species.kingdom.codon.{best}.{k}'] = v
errors = cc[cc['y_true'] != cc['y_pred']]
nested = errors['y_true'].isin(NESTED) & errors['y_pred'].isin(NESTED)
headline['errors.species.kingdom.codon.best.nested_vertebrate_share'] = errors.loc[nested, 'count'].sum() / errors['count'].sum()
print(f'best model: {best}; {errors["count"].sum()} errors, '
      f'{headline["errors.species.kingdom.codon.best.nested_vertebrate_share"]:.0%} of them between the four vertebrate divisions')
pd.Series(np.diag(share), index=KINGDOMS, name='recall').round(3).sort_values()

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 5.8))
im = ax.imshow(share.to_numpy(), cmap='Blues', vmin=0, vmax=1)
for i in range(len(KINGDOMS)):
    for j in range(len(KINGDOMS)):
        if share.iat[i, j] >= 0.005:
            ax.text(j, i, f'{share.iat[i, j]:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if share.iat[i, j] > 0.6 else '#0b0b0b')
ax.set_xticks(range(len(KINGDOMS)), KINGDOMS, rotation=45, ha='right', fontsize=8.5)
ax.set_yticks(range(len(KINGDOMS)), KINGDOMS, fontsize=8.5)
ax.set_xlabel('predicted')
ax.set_ylabel('true CUTG division')
ax.set_title(f'Kingdom confusion, {LABEL[best]}, codon features, species held out', loc='left', fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label='share of true class')
fig.tight_layout()
fig.savefig(FIG / 'kingdom_confusion.png', dpi=150)
plt.show()

Recall ranges from about 0.96 (bacteria, viruses, vertebrates) down to about 0.78–0.80 (primates, rodents and bacteriophages). Primates and rodents are misclassified mostly as 'mammal', and mammals as rodent, primate or vertebrate; but those four divisions are nested filing categories (primate and rodent within mammal, mammal within vertebrate), not separate branches, and confusions among them account for 23% of all this model's errors. Part of the kingdom error, then, is a property of the labels rather than a failure of the model; a primate mitochondrion misfiled as 'mammal' is hardly a biological mistake. Bacteriophages, for their part, are confused mostly with bacteria, whose codon usage phages tend to resemble, having evolved to be translated by their hosts.

## Tuning choices and data counts

In [ ]:
params = pd.read_csv(RES / 'chosen_params.csv')
knn = params[params['model'] == 'knn'].copy()
knn['k'] = knn['params'].map(lambda s: ast.literal_eval(s)['kneighborsclassifier__n_neighbors'])
for scheme, d in knn.groupby('scheme'):
    headline[f'knn.n_searches.{scheme}'] = len(d)
    for k, n in d['k'].value_counts().items():
        headline[f'knn.k_chosen.{scheme}.{k}'] = int(n)
pd.crosstab(knn['scheme'], knn['k'])

In [ ]:
raw = codon_utils.read_raw('../data/raw/codon_usage.csv')
df, row_counts = codon_utils.load_project('../data/raw/codon_usage.csv')
headline.update({
    'data.n_raw': len(raw),
    'data.n_repaired': int(raw['repaired'].sum()),
    'data.n_final': len(df),
    'data.n_removed': int(row_counts['removed'].sum()),
    'data.n_species_groups': df['group_species'].nunique(),
    'data.n_genus_groups': df['group_genus'].nunique(),
    'data.largest_genus_group': int(df['group_genus'].value_counts().iloc[0]),
})
row_counts

k-NN overwhelmingly prefers the smallest neighbourhoods: with species held out, k = 1 was chosen in 36 of the 60 inner searches, k = 3 in 23, and k = 7 once. In other words, the best k-NN model is very nearly "find the single most similar organism in the training data, and copy its label", which is why it is the model most exposed to relatedness, and why it suffers most when genus groups are held out. The data counts below record the row filter and the groups.

## The "stop" group and the genetic code

In animal mitochondria UGA encodes tryptophan, so the standard-code "stop" group carries a strong mitochondrial signal on its own (see `02_eda`). Do the models depend on it? `04_modeling` refit k-NN and XGBoost for DNA type, species held out, with all 21 codon groups, with the 20 groups other than "stop", and with "stop" alone.

In [ ]:
ablation = pd.read_csv(RES / 'stop_ablation.csv')
for (model, features), r in ablation.groupby(['model', 'features'])['f1_macro'].agg(['mean', 'std']).iterrows():
    headline[f'stop_ablation.dna_type.{model}.{features}.mean'] = r['mean']
    headline[f'stop_ablation.dna_type.{model}.{features}.sd'] = r['std']
for model, sub in ablation.groupby('model', sort=False):
    x, y = paired(sub, 'features', 'amino', 'amino_minus_stop')
    for k, v in compare(x, y, 'f1_macro').items():
        headline[f'stop_ablation.dna_type.{model}.amino_vs_minus_stop.{k}'] = v

# the genetic-code facts the README quotes (rows after the class filter)
animal = ['vertebrate', 'mammal', 'rodent', 'primate', 'invertebrate']
animal_mito = df[(df['dna_type'] == 'mitochondrial') & df['kingdom'].isin(animal)]
nuclear = df[df['dna_type'] == 'genomic']
headline.update({
    'gcode.n_animal_mito': len(animal_mito),
    'gcode.share_animal_mito': len(animal_mito) / len(df),
    'gcode.uga_over_ugg.animal_mito': (animal_mito['UGA'] > animal_mito['UGG']).mean(),
    'gcode.uga_over_ugg.nuclear': (nuclear['UGA'] > nuclear['UGG']).mean(),
})
table = ablation.groupby(['model', 'features'], sort=False)['f1_macro'].agg(['mean', 'std']).unstack('features')
table.round(3)

Dropping the "stop" group changes DNA-type macro-F1 by less than 0.001 for both models (corrected p = 0.77), while "stop" alone reaches 0.73 with k-NN and 0.79 with XGBoost, against 0.28 for the majority class. So the group does carry a strong mitochondrial signal by itself; but the other groups carry the same signal, through the other codons whose meaning changes in mitochondria (tryptophan's UGG, isoleucine's AUA, arginine's AGA and AGG), and the models don't depend on any one of them. Grouping codons by the standard code preserves this signature of the mitochondrial code, which is part of the reason the 21 groups come so close to the 64 codons on DNA type.

## results.json

Every number the README quotes is written to this file under a stable key, and `tests/test_readme_numbers.py` checks each number in the README against it, at the precision printed. The 2022 results were assembled by hand, from tables merged by position, and one bad merge attached the wrong metrics to the wrong models; this way, a number in the README can only come from the code that computed it.

In [ ]:
out = {k: (round(float(v), 6) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, np.integer) else v)
       for k, v in sorted(headline.items())}
(RES / 'results.json').write_text(json.dumps(out, indent=1) + '\n')
print(len(out), 'values written to', RES / 'results.json')